# CS229 L09 — Unsupervised Learning: K-Means and Gaussian Mixture Models

**Video:** Spring 2026 · [YouTube](https://www.youtube.com/watch?v=bSmIGBCoffA)  
**Instructor:** Tengyu Ma  
**Topics:** K-Means · Convergence · NP-hardness · K-means++ · GMM · Latent variables · EM algorithm · Jensen's inequality

---

> 📌 *Lecture:* "The reason pedagogically I like this section is it forces you to wrestle with some pretty basic questions in machine learning — what are we modeling? How are we putting structure into our models? What are we pulling out of it?"

---

## Roadmap

| Section | Key Idea |
|---|---|
| 1. Supervised vs unsupervised | Labels define the problem; without them, structure is underspecified |
| 2. K-Means | Hard assignment · iterative · monotone distortion decrease |
| 3. K-Means properties | NP-hard globally · local minima · K-means++ initialization |
| 4. Choosing K | Modeling question — no clean automatic answer |
| 5. GMM model | Soft assignment · latent variable · forward generative model |
| 6. EM algorithm | E-step (Bayes rule) · M-step (weighted MLE) |
| 7. K-Means as GMM limit | Hard assignment = soft assignment when W → {0,1} |
| 8. Jensen's inequality | Definition of convexity · key tool for EM convergence proof |
| 9. EM as MLE | Surrogate L_T(θ) · monotone likelihood increase |


## 1. Supervised vs Unsupervised Learning

### Supervised (L01–L08)
Given: $(x^{(1)}, y^{(1)}), \ldots, (x^{(n)}, y^{(n)})$  
Goal: learn $h_\theta: x \to y$  
Labels $y$ pin down what we want — the positive/negative class defines a canonical boundary.

### Unsupervised
Given: $x^{(1)}, \ldots, x^{(n)}$ — **no labels**  
Goal: discover hidden structure — clusters, density, manifolds

**Why harder:**
- Without labels, "correct" is not well-defined
- Must make stronger modeling assumptions
- Accept weaker guarantees — no convergence to global optimum in general

> 📌 *Lecture:* "In a formal sense, the K-Means problem is NP-hard — it's more computationally challenging. We've dropped some very important structure: we don't know the positive/negative class labels."

### When unsupervised makes sense
- You believe the data has **latent clusters** (gene expression types, astronomical sources, customer segments)
- You want a **descriptive feel** for your data before modeling
- You can **validate clusters** with auxiliary information

If your data has no structure (random spray of points), K-Means will return something — but it won't be meaningful.


## 2. K-Means Algorithm

### Setup

**Given:** points $\{x^{(1)}, \ldots, x^{(n)}\} \subset \mathbb{R}^d$, integer $K$  
**Find:** cluster assignments $c^{(i)} \in \{1, \ldots, K\}$ and centroids $\mu_1, \ldots, \mu_K \in \mathbb{R}^d$

### Objective: minimize distortion

$$J(c, \mu) = \sum_{i=1}^n \left\| x^{(i)} - \mu_{c^{(i)}} \right\|^2$$

Sum of squared distances from each point to its assigned centroid. K-Means minimizes this iteratively.

### Algorithm

```
1. Randomly initialize μ_1, ..., μ_K
2. Repeat until assignments don't change:
   a. ASSIGN: for each i, c(i) = argmin_j ||x(i) - μ_j||²
   b. UPDATE:  for each j, μ_j = mean of {x(i) : c(i) = j}
```

### Why it terminates

Each step of K-Means can only **decrease or maintain** $J$:
- **Assign step:** reassigning $x^{(i)}$ to its nearest centroid can only decrease $\|x^{(i)} - \mu_{c^{(i)}}\|^2$
- **Update step:** the mean minimizes sum of squared distances — replacing $\mu_j$ with the centroid of its cluster can only decrease $J$

$$J(c^{(t+1)}, \mu^{(t+1)}) \leq J(c^{(t)}, \mu^{(t)})$$

$J$ is bounded below by 0 and decreases monotonically → algorithm terminates.

> 📌 *Lecture:* "Underneath the covers, the L2 distortion functional is monotonically decreasing. Over time that average L2 distance is shrinking — but it doesn't say the rate at which it's shrinking."

### Stopping criteria (in practice)
1. Assignments unchanged between iterations (theoretical)
2. Fixed number of iterations (practical — most common)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

np.random.seed(42)

# --- Generate synthetic clustered data ---
def make_blobs(centers, std=0.6, n_per=80):
    X = np.vstack([np.random.randn(n_per, 2) * std + c for c in centers])
    return X

centers_true = [[-2, -2], [2, -2], [0, 2.5]]
X = make_blobs(centers_true)

# --- K-Means from scratch ---
def kmeans(X, K, max_iter=100, seed=0):
    rng = np.random.default_rng(seed)
    # Random initialization
    idx = rng.choice(len(X), K, replace=False)
    mu = X[idx].copy()
    history = [mu.copy()]
    distortions = []

    for _ in range(max_iter):
        # Assign step: c(i) = argmin_j ||x(i) - mu_j||^2
        dists = np.linalg.norm(X[:, None] - mu[None], axis=2)  # (n, K)
        c = np.argmin(dists, axis=1)

        # Distortion
        J = sum(np.sum((X[c == j] - mu[j])**2) for j in range(K) if np.any(c == j))
        distortions.append(J)

        # Update step: mu_j = mean of cluster j
        mu_new = np.array([X[c == j].mean(axis=0) if np.any(c == j) else mu[j] for j in range(K)])

        if np.allclose(mu, mu_new):   # converged
            break
        mu = mu_new
        history.append(mu.copy())

    return c, mu, distortions, history

K = 3
c, mu, distortions, history = kmeans(X, K)

# --- Plot ---
colors = ['#e74c3c', '#3498db', '#2ecc71']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for j in range(K):
    axes[0].scatter(X[c == j, 0], X[c == j, 1], c=colors[j], alpha=0.6, s=30, label=f'Cluster {j+1}')
axes[0].scatter(mu[:, 0], mu[:, 1], c='black', s=200, marker='*', zorder=5, label='Centroids')
axes[0].set_title(f'K-Means result (K={K}, {len(distortions)} iterations)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(distortions, 'o-', color='#e67e22', lw=2)
axes[1].set_title('Distortion J vs iteration (monotonically decreasing)')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('J = Σ||x - μ_c||²')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Converged in {len(distortions)} iterations. Final J = {distortions[-1]:.2f}")

## 3. K-Means Properties

### NP-Hardness

Finding the **global minimum** of $J(c, \mu)$ over all possible assignments is NP-hard.  
K-Means converges to a **local minimum** — the solution depends on initialization.

> 📌 *Lecture:* "It picks not the right answer. Both decreasing so slowly (exponential steps) or finding not the right answer are possible."

### Local minima and initialization

Because the algorithm is deterministic after initialization, different random starts → different solutions. This is why:
- **Run multiple times** with different seeds and take the best $J$
- Or use a smarter initialization — K-Means++

### K-Means++ initialization

Developed by David Arthur and Sergei Vassilvitskii (Stanford grad students). Provably achieves an $O(\log K)$ approximation ratio to the global optimum.

**Algorithm:**
1. Choose first centroid $\mu_1$ uniformly at random
2. For each subsequent centroid: choose $x^{(i)}$ with probability proportional to $\min_j \|x^{(i)} - \mu_j\|^2$ — points far from existing centroids are more likely to be chosen
3. Repeat until $K$ centroids are placed
4. Run standard K-Means from this initialization

**This is the default in `sklearn.cluster.KMeans`** (`init='k-means++'`).

---

> 🎯 **Interview:** Why does K-Means not find the global optimum, and how do you handle this in practice?
>
> **A:** K-Means minimizes the distortion objective J iteratively via coordinate descent — each step decreases J but the algorithm can get stuck in a local minimum, not the global one. This is because finding the global optimum of J is NP-hard. In practice: (1) use K-means++ initialization (default in sklearn) which gives an O(log K) approximation guarantee; (2) run multiple random restarts and keep the solution with lowest J; (3) choose K carefully — often the local minimum at the right K is good enough for the application.

In [ ]:
# --- K-Means++ initialization vs random: compare final distortion across seeds ---
from sklearn.cluster import KMeans

n_runs = 20
J_random  = [KMeans(n_clusters=3, init='random',    n_init=1, random_state=s).fit(X).inertia_ for s in range(n_runs)]
J_kpp     = [KMeans(n_clusters=3, init='k-means++', n_init=1, random_state=s).fit(X).inertia_ for s in range(n_runs)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(J_random, 'o-', label='Random init',    color='#e74c3c', alpha=0.8)
ax.plot(J_kpp,    's-', label='K-Means++ init', color='#3498db', alpha=0.8)
ax.set_xlabel('Random seed'); ax.set_ylabel('Final distortion J')
ax.set_title('K-Means++ vs random initialization — final distortion across 20 runs')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Random init  — mean J: {np.mean(J_random):.1f}, std: {np.std(J_random):.1f}")
print(f"K-Means++ — mean J: {np.mean(J_kpp):.1f},    std: {np.std(J_kpp):.1f}")
print("K-Means++ is more consistent — fewer bad local minima.")

## 4. Choosing K

### The problem

K is a **modeling assumption** — you assert there are K clusters in your data. There is no single automatic method to choose K from the data alone.

### The elbow method

Plot distortion $J$ vs $K$. As $K$ increases, $J$ always decreases (more clusters → tighter fit). Look for an "elbow" — a point where the rate of decrease changes sharply.

> 📌 *Lecture:* "As you increase K you're getting finer and finer clusters, but is it really better? I don't know. This is kind of the real challenge when you start to do more unsupervised modeling."

**Limitation:** The elbow is often ambiguous. GMMs provide a better criterion via likelihood (AIC/BIC) — covered next.

### In practice
- Domain knowledge: "I know there are 3 cell types" → K=3
- Try multiple K, validate with auxiliary information (biology, metadata, downstream task)
- GMM + BIC for a principled model-based selection


In [ ]:
# --- Elbow plot ---
Ks = range(1, 9)
distortions_k = [KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=0).fit(X).inertia_ for k in Ks]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(Ks, distortions_k, 'o-', color='#8e44ad', lw=2)
ax.axvline(3, color='gray', ls='--', alpha=0.6, label='True K=3')
ax.set_xlabel('K'); ax.set_ylabel('Distortion J')
ax.set_title('Elbow plot — distortion vs number of clusters')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("Elbow at K=3 is visible here, but in practice it is often ambiguous.")

## 5. Gaussian Mixture Model

### Motivation: soft assignment

K-Means makes **hard assignments** — each point belongs to exactly one cluster. Points near a boundary are assigned arbitrarily. GMM replaces this with **soft assignment**: probability of belonging to each cluster.

### Generative model (forward model)

To generate data point $x^{(i)}$:
1. **Pick a source:** $z^{(i)} \sim \text{Multinomial}(\phi_1, \ldots, \phi_K)$ — choose cluster $j$ with probability $\phi_j$
2. **Sample from that cluster:** $x^{(i)} \mid z^{(i)} = j \sim \mathcal{N}(\mu_j, \Sigma_j)$

**Parameters to learn:** $\theta = \{\phi_j, \mu_j, \Sigma_j\}_{j=1}^K$

| Parameter | Meaning | Shape |
|---|---|---|
| $\phi_j$ | Mixture weight (prior probability of cluster $j$) | scalar, $\sum_j \phi_j = 1$ |
| $\mu_j$ | Mean of cluster $j$ | $\mathbb{R}^d$ |
| $\Sigma_j$ | Covariance of cluster $j$ | $\mathbb{R}^{d\times d}$ |

### Latent variable

$z^{(i)}$ is the **latent (hidden) variable** — we believe it exists (which cluster generated $x^{(i)}$) but we never observe it directly.

$$p(x^{(i)}, z^{(i)} = j; \theta) = p(z^{(i)} = j; \phi) \cdot p(x^{(i)} \mid z^{(i)} = j; \mu_j, \Sigma_j) = \phi_j \cdot \mathcal{N}(x^{(i)}; \mu_j, \Sigma_j)$$

Marginalizing over $z$:

$$p(x^{(i)}; \theta) = \sum_{j=1}^K \phi_j \cdot \mathcal{N}(x^{(i)}; \mu_j, \Sigma_j)$$

This is the **mixture of Gaussians** density.

### Soft assignment via Bayes rule

Given current parameters $\theta$, the posterior probability that $x^{(i)}$ came from cluster $j$:

$$w_j^{(i)} \triangleq p(z^{(i)} = j \mid x^{(i)}; \theta) = \frac{\phi_j \cdot \mathcal{N}(x^{(i)}; \mu_j, \Sigma_j)}{\sum_{\ell=1}^K \phi_\ell \cdot \mathcal{N}(x^{(i)}; \mu_\ell, \Sigma_\ell)}$$

This is just Bayes rule — numerator is the joint, denominator normalizes.

> 📌 *Lecture:* "If cluster two generated 100 times more data than cluster one, you'd say this point came from cluster two. That inference you just did is Bayes rule."

---

> 🎯 **Interview:** What is the difference between K-Means and GMM?
>
> **A:** Both partition data into K groups, but they differ in three ways. First, assignment: K-Means makes hard assignments (each point belongs to exactly one cluster), GMM makes soft probabilistic assignments (each point has a probability of belonging to each cluster). Second, geometry: K-Means uses Euclidean distance so clusters are implicitly spherical; GMM models each cluster as a Gaussian with a full covariance matrix, so clusters can be elongated, tilted ellipses. Third, uncertainty: K-Means gives no measure of uncertainty for ambiguous points; GMM gives a posterior probability. K-Means is the limiting case of GMM when the cluster covariances collapse to identity (hard assignment limit).

## 6. EM Algorithm for GMM

We cannot run MLE directly because $z^{(i)}$ is unobserved. EM alternates between:
- **E-step:** estimate the latent variables (compute $w_j^{(i)}$)
- **M-step:** maximize parameters given the estimated latents

### E-step (Expectation)

For each $i = 1, \ldots, n$ and $j = 1, \ldots, K$:

$$w_j^{(i)} = \frac{\phi_j \cdot \mathcal{N}(x^{(i)}; \mu_j, \Sigma_j)}{\sum_{\ell=1}^K \phi_\ell \cdot \mathcal{N}(x^{(i)}; \mu_\ell, \Sigma_\ell)}$$

### M-step (Maximization)

Given weights $w_j^{(i)}$, update parameters:

$$\phi_j = \frac{1}{n} \sum_{i=1}^n w_j^{(i)} \qquad \text{(effective fraction of points in cluster } j\text{)}$$

$$\mu_j = \frac{\sum_{i=1}^n w_j^{(i)} x^{(i)}}{\sum_{i=1}^n w_j^{(i)}} \qquad \text{(weighted mean)}$$

$$\Sigma_j = \frac{\sum_{i=1}^n w_j^{(i)} (x^{(i)} - \mu_j)(x^{(i)} - \mu_j)^T}{\sum_{i=1}^n w_j^{(i)}} \qquad \text{(weighted covariance)}$$

### Connection to K-Means

If $w_j^{(i)} \in \{0, 1\}$ (hard assignment), the M-step reduces to:
- $\phi_j = $ fraction of points in cluster $j$
- $\mu_j = $ mean of points in cluster $j$

This is exactly K-Means. **K-Means = GMM + hard assignment + spherical equal-variance clusters.**

> 📌 *Lecture:* "If these were zeros and ones, convince yourself that this would exactly be the same mean we were computing in K-Means — the centroid."


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

np.random.seed(0)

# --- GMM from scratch (1D for clarity, then 2D) ---

def gaussian_pdf(x, mu, sigma2):
    """1D Gaussian density."""
    return np.exp(-0.5 * (x - mu)**2 / sigma2) / np.sqrt(2 * np.pi * sigma2)

def gaussian_pdf_2d(X, mu, Sigma):
    """2D Gaussian density for each row of X."""
    d = X.shape[1]
    diff = X - mu
    inv_S = np.linalg.inv(Sigma)
    exponent = -0.5 * np.sum(diff @ inv_S * diff, axis=1)
    norm = np.sqrt((2 * np.pi)**d * np.linalg.det(Sigma))
    return np.exp(exponent) / norm

def fit_gmm(X, K, n_iter=50, seed=1):
    rng = np.random.default_rng(seed)
    n, d = X.shape

    # Initialize
    phi = np.ones(K) / K
    idx = rng.choice(n, K, replace=False)
    mu = X[idx].copy().astype(float)
    Sigma = np.array([np.eye(d) * 2.0 for _ in range(K)])

    log_likelihoods = []

    for _ in range(n_iter):
        # E-step: compute w_ij = p(z=j | x_i)
        W = np.zeros((n, K))
        for j in range(K):
            W[:, j] = phi[j] * gaussian_pdf_2d(X, mu[j], Sigma[j])
        W_sum = W.sum(axis=1, keepdims=True)
        W_sum = np.where(W_sum == 0, 1e-300, W_sum)
        W /= W_sum

        # Log-likelihood
        ll = np.log(W_sum + 1e-300).sum()
        log_likelihoods.append(ll)

        # M-step: update parameters
        Nj = W.sum(axis=0)  # effective count per cluster
        phi = Nj / n
        mu = (W.T @ X) / Nj[:, None]
        for j in range(K):
            diff = X - mu[j]
            Sigma[j] = (W[:, j:j+1] * diff).T @ diff / Nj[j] + np.eye(d) * 1e-6

    return phi, mu, Sigma, W, log_likelihoods

# Run GMM on 2D data
K = 3
phi, mu_gmm, Sigma, W, lls = fit_gmm(X, K)
labels_gmm = W.argmax(axis=1)  # hard labels for plotting

# Confidence ellipse helper
def plot_ellipse(ax, mu, Sigma, color, n_std=2.0):
    vals, vecs = np.linalg.eigh(Sigma)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=mu, width=w, height=h, angle=angle,
                  edgecolor=color, fc='none', lw=2.5, linestyle='--')
    ax.add_patch(ell)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71']
for j in range(K):
    mask = labels_gmm == j
    axes[0].scatter(X[mask, 0], X[mask, 1], c=colors[j], alpha=0.5, s=25)
    axes[0].scatter(*mu_gmm[j], c='black', s=150, marker='*', zorder=5)
    plot_ellipse(axes[0], mu_gmm[j], Sigma[j], colors[j])

axes[0].set_title('GMM — soft clusters with covariance ellipses (2σ)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(lls, 'o-', color='#8e44ad', lw=2)
axes[1].set_title('Log-likelihood vs EM iteration (monotonically increasing)')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Log-likelihood')
axes[1].grid(True, alpha=0.3)

plt.suptitle('GMM fitted with EM — from scratch', fontsize=12)
plt.tight_layout(); plt.show()
print(f"Mixture weights φ: {phi.round(3)}")
print(f"Cluster means:\n{mu_gmm.round(2)}")

In [ ]:
# --- Compare K-Means vs GMM side by side ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# K-Means
for j in range(K):
    mask = c == j
    axes[0].scatter(X[mask, 0], X[mask, 1], c=colors[j], alpha=0.5, s=25)
axes[0].scatter(mu[:, 0], mu[:, 1], c='black', s=200, marker='*', zorder=5)
axes[0].set_title('K-Means (hard assignment)')
axes[0].grid(True, alpha=0.3)

# GMM — color by soft assignment (blend colors by W)
rgb = np.array([[0.91, 0.30, 0.24], [0.20, 0.60, 0.86], [0.18, 0.80, 0.44]])
point_colors = W @ rgb  # weighted blend
axes[1].scatter(X[:, 0], X[:, 1], c=point_colors, s=25, alpha=0.8)
for j in range(K):
    axes[1].scatter(*mu_gmm[j], c='black', s=150, marker='*', zorder=5)
    plot_ellipse(axes[1], mu_gmm[j], Sigma[j], colors[j])
axes[1].set_title('GMM (soft assignment — color = mixture of cluster colors)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('K-Means vs GMM — ambiguous points handled differently', fontsize=11)
plt.tight_layout(); plt.show()
print("Ambiguous boundary points appear blended in GMM — reflecting genuine uncertainty.")

## 7. Jensen's Inequality

### Convex functions

A function $f: \mathbb{R} \to \mathbb{R}$ is **convex** if for all $a, b$ and $\lambda \in [0,1]$:

$$f(\lambda a + (1-\lambda)b) \leq \lambda f(a) + (1-\lambda)f(b)$$

Geometrically: the **chord** between any two points on the graph lies **above** the graph.

Equivalently (for twice-differentiable $f$): $f''(x) \geq 0$ for all $x$.

**Canonical examples:** $f(x) = x^2$, $f(x) = e^x$, $f(x) = |x|$

### Jensen's Inequality

For a **convex** function $f$ and random variable $X$:

$$\boxed{f(\mathbb{E}[X]) \leq \mathbb{E}[f(X)]}$$

The function of the expectation ≤ the expectation of the function.

> 📌 *Lecture:* "Jensen's inequality IS the definition of convexity. It's like the whole reason we're after it. Sometimes it's presented as some mysterious consequence — it is the definition of convexity."

### Concave version (what EM actually uses)

$f$ is **concave** if $-f$ is convex. For concave $f$ (e.g., $f = \log$), Jensen reverses:

$$\boxed{f(\mathbb{E}[X]) \geq \mathbb{E}[f(X)]}$$

For **log** specifically:

$$\log\left(\mathbb{E}[X]\right) \geq \mathbb{E}[\log X]$$

This allows us to move a log **inside** an expectation at the cost of an inequality — exactly what EM needs to construct the surrogate $L_T(\theta)$.


In [ ]:
# --- Visualize Jensen's inequality ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(-2, 3, 300)

# Convex: f(x) = x^2
f_conv = x**2
a, b, lam = -1.5, 2.5, 0.4
E_X = lam * a + (1-lam) * b
E_fX = lam * a**2 + (1-lam) * b**2
fEX = E_X**2

axes[0].plot(x, f_conv, 'b', lw=2, label='f(x) = x²')
axes[0].plot([a, b], [a**2, b**2], 'r--', lw=2, label='chord')
axes[0].scatter([E_X], [fEX],  color='green', s=120, zorder=5, label=f'f(E[X]) = {fEX:.2f}')
axes[0].scatter([E_X], [E_fX], color='red',   s=120, zorder=5, marker='s', label=f'E[f(X)] = {E_fX:.2f}')
axes[0].annotate('', xy=(E_X, E_fX), xytext=(E_X, fEX),
                 arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
axes[0].set_title('Convex: f(E[X]) ≤ E[f(X)]  (Jensen)')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-2.2, 3.2)

# Concave: f(x) = log(x)
x_pos = np.linspace(0.1, 4, 300)
f_conc = np.log(x_pos)
a2, b2, lam2 = 0.3, 3.5, 0.5
E_X2 = lam2 * a2 + (1-lam2) * b2
E_fX2 = lam2 * np.log(a2) + (1-lam2) * np.log(b2)
fEX2 = np.log(E_X2)

axes[1].plot(x_pos, f_conc, 'b', lw=2, label='f(x) = log(x)')
axes[1].plot([a2, b2], [np.log(a2), np.log(b2)], 'r--', lw=2, label='chord')
axes[1].scatter([E_X2], [fEX2],  color='green', s=120, zorder=5, label=f'f(E[X]) = {fEX2:.2f}')
axes[1].scatter([E_X2], [E_fX2], color='red',   s=120, zorder=5, marker='s', label=f'E[f(X)] = {E_fX2:.2f}')
axes[1].annotate('', xy=(E_X2, E_fX2), xytext=(E_X2, fEX2),
                 arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
axes[1].set_title('Concave (log): f(E[X]) ≥ E[f(X)]  (Jensen reversed)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle("Jensen's Inequality — geometric picture", fontsize=12)
plt.tight_layout(); plt.show()

## 8. EM as Maximum Likelihood

### The log-likelihood

$$\ell(\theta) = \sum_{i=1}^n \log p(x^{(i)}; \theta) = \sum_{i=1}^n \log \sum_{j=1}^K \phi_j \cdot \mathcal{N}(x^{(i)}; \mu_j, \Sigma_j)$$

The **log of a sum** has no closed-form maximizer — the latent $z$ is inside the sum.

### The EM idea: surrogate lower bound

At each iteration $t$, EM constructs a **surrogate** $L_t(\theta)$ that:
1. Is a lower bound: $L_t(\theta) \leq \ell(\theta)$ for all $\theta$
2. Is tight at $\theta^{(t)}$: $L_t(\theta^{(t)}) = \ell(\theta^{(t)})$
3. Is easier to maximize (log of a product, not log of a sum)

**E-step** builds $L_t$ using the current $\theta^{(t)}$ and Jensen's inequality (concave log).  
**M-step** maximizes $L_t(\theta)$ to get $\theta^{(t+1)}$.

### Monotone likelihood increase

$$\ell(\theta^{(t+1)}) \geq L_t(\theta^{(t+1)}) \geq L_t(\theta^{(t)}) = \ell(\theta^{(t)})$$

Every EM iteration increases (or maintains) the log-likelihood. EM **cannot decrease** $\ell$.

### Does it find the global maximum?

No — same issue as K-Means. EM converges to a **local maximum** of $\ell(\theta)$. Multiple random restarts + pick the run with highest $\ell$ at convergence.

> 📌 *Lecture:* "Do you think this algorithm finds a global maximum? Most certainly not. When we reduce to the 0-1 case we saw that was a problem for K-Means. We'd expect the same thing here."

---

> 🎯 **Interview:** Why does EM always increase the log-likelihood?
>
> **A:** At each iteration, EM constructs a surrogate objective $L_t(\theta)$ using Jensen's inequality (applied to the concave log function) that lower-bounds the true log-likelihood everywhere and is tight at the current parameters $\theta^{(t)}$. The M-step maximizes $L_t$ to get $\theta^{(t+1)}$, which can only increase $L_t$. Since $L_t$ is a lower bound on $\ell$ and they agree at $\theta^{(t)}$, this guarantees $\ell(\theta^{(t+1)}) \geq L_t(\theta^{(t+1)}) \geq L_t(\theta^{(t)}) = \ell(\theta^{(t)})$. So the true log-likelihood never decreases. This is why watching the log-likelihood during EM is a useful diagnostic — if it ever decreases, there is a bug.

## Summary

### K-Means vs GMM

| | K-Means | GMM |
|---|---|---|
| Assignment | Hard ($c^{(i)} \in \{1,\ldots,K\}$) | Soft ($w_j^{(i)} \in [0,1]$) |
| Cluster shape | Spherical (Euclidean distance) | Elliptical (full covariance) |
| Objective | Minimize distortion $J$ | Maximize log-likelihood $\ell$ |
| Convergence | Monotone $J$ decrease | Monotone $\ell$ increase |
| Global optimum | NP-hard | NP-hard (local max only) |
| Model selection | Elbow (weak) | AIC/BIC (principled) |
| Uncertainty | None | Posterior $w_j^{(i)}$ |

### EM in two lines

**E-step:** use Bayes rule + current parameters to compute soft assignments $w_j^{(i)} = p(z^{(i)}=j \mid x^{(i)}; \theta)$  
**M-step:** use weighted MLE with those soft assignments to update $\phi$, $\mu$, $\Sigma$

### Jensen's in one line

For concave $f$ (e.g., log): $f(\mathbb{E}[X]) \geq \mathbb{E}[f(X)]$ — the whole EM convergence proof rests on this.

---

## External Resources

| Resource | What to read |
|---|---|
| CS229 Notes Part 9 | K-Means, GMM, EM derivation |
| Bishop *PRML* Ch. 9 | Mixture models and EM — the standard reference |
| sklearn docs | `KMeans`, `GaussianMixture` — API + examples |
| Visualizing K-Means | [distill.pub](https://distill.pub) — interactive clustering demos |
